# 03 · De las señales a los resultados de las estrategias

Comparamos **comprar y mantener, momentum y reversión** durante 1927–1989. Utilizamos las señales anteriores y los componentes del notebook 02. Todavía no hay filtros de aprendizaje automático ni evaluación desde 1990.

Los resultados son una aproximación histórica sobre promedios de Shiller y dividendos reconstruidos. No representan operaciones ejecutables a esos precios ni costes históricos observados.

Necesitas Shiller, señales y componentes ya preparados.

## 1. Ver los supuestos y comprobar las entradas

Cada cuenta parte del mismo capital y busca una exposición del 100 % comprado, 100 % vendido o efectivo. En un mes se aplica la señal decidida en el anterior.

El coste de negociación se aplica al nominal que realmente cambia, incluyendo el reajuste de los cortos y el cierre final. El préstamo solo se cobra al estar vendido. Los dividendos se suman en largos y se deben en cortos. RF remunera el efectivo o el capital aportado al corto, sin interés adicional por el dinero obtenido de la venta corta. El largo no recibe RF adicional.

In [ ]:
from tfm.data import project_root, save_result
from tfm.simulation import simulate_baselines, summarize_baselines, WARNINGS as SIMULATION_WARNINGS
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from tfm.data import read_result
from tfm.simulation import STRATEGIES

ROOT = project_root()
parametros = {"initial_capital": 100.0, "transaction_cost_rate": 0.001,
              "annual_borrow_rate": 0.01, "include_dividends": True}
componentes, _ = read_result(ROOT, "monthly_return_components")
senales, _ = read_result(ROOT, "shiller_signals")
display(pd.DataFrame([
    {"supuesto": "Capital inicial normalizado", "valor": parametros["initial_capital"]},
    {"supuesto": "Negociación (% del nominal)", "valor": 100 * parametros["transaction_cost_rate"]},
    {"supuesto": "Préstamo (% anual, solo cortos)", "valor": 100 * parametros["annual_borrow_rate"]},
    {"supuesto": "Dividendos aproximados incluidos", "valor": parametros["include_dividends"]},
]))
print(f"Desarrollo: {componentes.period.iloc[0]} → {componentes.period.iloc[-1]}")
print("Se liquidan las posiciones al final. La evaluación desde 1990 sigue deshabilitada.")

## 2. Ejecutar el escenario central

**Esta celda guarda o sustituye** `data/processed/baseline_simulations.parquet` y su resumen JSON. No descarga fuentes ni modifica las reglas. Para estudiar otros supuestos, edita los supuestos centrales en los supuestos visibles del cuaderno y vuelve a ejecutar desde la primera celda. Los escenarios deben declararse y no elegirse después para favorecer un resultado.

El resultado incluye una fila por mes y estrategia: 756 meses × 3 estrategias = 2.268 filas. Los ingresos de una venta corta no se contabilizan como una ganancia. Una cuenta sin capital detiene la simulación.

In [ ]:
simulaciones = simulate_baselines(componentes, senales, parametros)
informe = {"status": "passed", "rows": len(simulaciones), "months_per_strategy": len(componentes),
          "first_period": componentes.period.iloc[0], "last_period": componentes.period.iloc[-1],
          "parameters": parametros, "liquidated_at_end": True,
          "strategies": summarize_baselines(simulaciones, parametros),
          "warnings": SIMULATION_WARNINGS}
resumen = save_result(ROOT, "baseline_simulations", simulaciones, informe)
print(f"Resultado: {informe['rows']} filas | {informe['months_per_strategy']} meses por estrategia")
print(resumen["path"])
metricas = pd.DataFrame(informe["strategies"]).set_index("label")
vista = metricas[["capital_final", "total_return", "annualized_return", "max_drawdown", "sharpe"]].copy()
for columna in ["total_return", "annualized_return", "max_drawdown"]:
    vista[columna] *= 100
vista = vista.rename(columns={"capital_final": "Capital final", "total_return": "Acumulada (%)",
                              "annualized_return": "Anualizada (%)", "max_drawdown": "Caída máxima (%)", "sharpe": "Sharpe"})
display(vista.round(4))

## 3. Entender un mes: mayo de 1975

En el notebook 01 vimos las señales para mayo de 1975 usando información hasta abril, momentum proponía efectivo y reversión una posición vendedora. Aquí observamos el resultado posterior.

Las columnas de la siguiente tabla son estrategias, por lo que cada fila es una parte de la cuenta. Los importes son unidades de capital normalizado. En 1975 las cuentas ya llevan años evolucionando y pueden tener capitales diferentes, es decir, para comparar el resultado de ese mes, mira también la rentabilidad neta porcentual.

Una posición corta puede necesitar negociar aunque siga siendo −1, porque el precio y el capital cambiaron. Los largos reinvierten dividendos sin comisión separada. Los costes de cierre final solo aparecen en el último mes de toda la simulación.

In [ ]:
MES_OBJETIVO = "1975-05"
mes = simulaciones.loc[simulaciones.period.eq(MES_OBJETIVO)].set_index("strategy")
if len(mes) != len(STRATEGIES):
    raise ValueError("Elige un mes dentro del desarrollo preparado.")
columnas = {
    "position": "Posición (+1, 0, −1)", "capital_start": "Capital al empezar el mes",
    "transaction_cost": "Coste de negociación (se resta)", "price_pnl": "Resultado del precio",
    "dividend_pnl": "Dividendo aproximado (con signo)", "cash_interest": "Remuneración RF",
    "borrow_cost": "Préstamo (se resta)", "liquidation_cost": "Cierre final (se resta)",
    "capital_end": "Capital al terminar el mes",
}
print(f"Decisión: {mes.decision_period.iloc[0]} | Resultado: {MES_OBJETIVO}")
display(mes[list(columnas)].rename(index=STRATEGIES, columns=columnas).T.round(6))
display(pd.DataFrame({"Rentabilidad neta (%)": 100 * mes.net_return}).rename(index=STRATEGIES).round(6))
balance = (mes.capital_start - mes.transaction_cost + mes.price_pnl + mes.dividend_pnl
           + mes.cash_interest - mes.borrow_cost - mes.liquidation_cost)
assert (balance - mes.capital_end).abs().max() < 1e-9

## 4. Evolución del capital y caídas desde máximos

Las tres curvas parten del capital inicial. El gráfico inferior muestra la caída respecto al máximo anterior, incluyendo ese capital inicial, un primer mes con pérdidas también cuenta como caída.

Puedes activar `ESCALA_LOGARITMICA` para que distancias verticales iguales representen cambios proporcionales iguales. Solo cambia la presentación del gráfico, no los cálculos.

In [ ]:
ESCALA_LOGARITMICA = False
evolucion = simulaciones.pivot(index="period", columns="strategy", values="capital_end")
mes_inicial = str(pd.Period(evolucion.index[0], freq="M") - 1)
inicial = pd.DataFrame({nombre: [parametros["initial_capital"]] for nombre in STRATEGIES}, index=[mes_inicial])
evolucion = pd.concat([inicial, evolucion]).sort_index()
fechas = pd.PeriodIndex(evolucion.index, freq="M").to_timestamp(how="end")
caidas = 100 * (evolucion / evolucion.cummax() - 1)

fig, ejes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, layout="constrained")
for nombre, estilo in zip(STRATEGIES, ["-", "--", ":"]):
    ejes[0].plot(fechas, evolucion[nombre], linestyle=estilo, label=STRATEGIES[nombre])
    ejes[1].plot(fechas, caidas[nombre], linestyle=estilo, label=STRATEGIES[nombre])
if ESCALA_LOGARITMICA:
    ejes[0].set_yscale("log")
ejes[0].set(title="Estrategias básicas · desarrollo 1927–1989", ylabel="Capital normalizado")
ejes[1].set(ylabel="Caída desde máximos (%)", xlabel="Año")
for eje in ejes:
    eje.grid(alpha=0.25)
    eje.legend()
plt.show()

## 5. Cómo interpretar la comparación

El capital final y la rentabilidad acumulada muestran cuánto ha cambiado cada cuenta bajo estos supuestos. La anualizada expresa una tasa compuesta equivalente y la caída máxima refleja la peor pérdida desde un máximo anterior. El Sharpe utiliza el exceso mensual neto sobre RF y su dispersión, anualizado por raíz de doce. Si esa dispersión es nula, no se asigna un Sharpe.

Esta comparación establece las referencias sin filtrar. Un buen resultado aquí no demuestra predictibilidad ni que un modelo vaya a mejorarla. Antes de usar el tramo desde 1990 se completarán las alternativas sencillas, las variables, las etiquetas, la selección y los controles de sobreajuste.

El notebook 04 compara préstamo, negociación y dividendos. La sensibilidad al retraso de las señales sigue pendiente.